In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

import re
import json

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, AutoModelForMaskedLM
import torch

import gensim
from tqdm import tqdm

import requests

import faiss
import os

from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [10]:
import pinecone
from langchain.vectorstores import Pinecone

In [2]:
biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
biobert_model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

In [4]:
llama3_model_name = "meta-llama/Llama-3.2-3B-Instruct"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, use_auth_token="HUGGINGFACE-ACCESS_TOKEN")
llama3_model = AutoModelForCausalLM.from_pretrained(
    llama3_model_name,
    torch_dtype=torch.float16,  # Use float16 for efficiency
    device_map="auto",  # Automatically allocate to available GPU
    use_auth_token="HUGGINGFACE-ACCESS_TOKEN"
)
llama3_tokenizer.pad_token = llama3_tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
!nvidia-smi

Wed Apr 23 15:43:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           On  |   00000000:88:00.0 Off |                    0 |
| N/A   27C    P0             34W /  250W |    6484MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
notes = pd.read_csv('/N/project/ADRD/Mohit/MIMIC/discharge.csv.gz')

In [7]:
icd_diag = pd.read_csv('/N/project/ADRD/Mohit/MIMIC/d_icd_diagnoses.csv.gz')

# Search for multiple words in the long_title column
search_words = ["dementia", "Alzheimer's"]
pattern = '|'.join(search_words)  # Create a regex pattern with "or" logic

# Filter rows containing any of the search words
result = icd_diag[icd_diag["long_title"].str.contains(pattern, case=False, na=False)]

diag_icd_code_list = result['icd_code'].to_list()

diagnoses_data = pd.read_csv('/N/project/ADRD/Mohit/MIMIC/diagnoses_icd.csv.gz')
diag_icd_code_list = result['icd_code'].to_list()

ad_diag_data = diagnoses_data[diagnoses_data['icd_code'].isin(diag_icd_code_list)]
notes_filtered = notes.merge(ad_diag_data, how = 'inner', on = ['subject_id', 'hadm_id'])

In [8]:
notes_sample = notes_filtered.sample(1000, random_state=42)

In [9]:
notes_sample.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text,seq_num,icd_code,icd_version
3571,11968246-DS-7,11968246,26241345,DS,7,2184-12-31 00:00:00,2185-01-10 12:19:00,\nName: ___ Unit No: _...,3,29420,9
16593,18991213-DS-16,18991213,20749681,DS,16,2204-08-08 00:00:00,2204-08-08 16:30:00,\nName: ___. Unit No: ___\n \nAd...,18,29410,9
16534,18960632-DS-12,18960632,24027650,DS,12,2130-05-10 00:00:00,2130-05-11 19:10:00,\nName: ___ Unit No: ___\n \n...,5,F0280,10
5245,12860165-DS-16,12860165,28209731,DS,16,2132-02-02 00:00:00,2132-02-02 15:06:00,\nName: ___ Unit No: ___\...,6,3310,9
16341,18858088-DS-11,18858088,20730586,DS,11,2194-09-28 00:00:00,2194-10-01 18:41:00,\nName: ___ Unit No: ___\n ...,8,29410,9


In [10]:
notes_sample[notes_sample['subject_id']==12784119]

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text,seq_num,icd_code,icd_version


In [11]:
class GenerateEmbedding:
    def __init__(self, notes_data, model_name, biobert_model, biobert_tokenizer, 
                 bluebert_model = None, bluebert_tokenizer = None, pubmed_model = None, pubmed_tokenizer = None):
        self.model_name = model_name
        self.data = notes_data
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Section patterns
        self.patterns = {
            "History of Present Illness": re.compile(r"History of Present Illness:\s*(.*?)(?:\nPast Medical History:|\n\Z)", re.DOTALL),
            "Past Medical History": re.compile(r"Past Medical History:\s*(.*?)(?:\nSocial History:|\n\Z)", re.DOTALL),
            "Family History": re.compile(r"Family History:\s*(.*?)(?:\nPhysical Exam:|\n\Z)", re.DOTALL),
            "Physical Exam": re.compile(r"Physical Exam:\s*(.*?)(?:\nPertinent Results:|\n\Z)", re.DOTALL),
            "Pertinent Results": re.compile(r"Pertinent Results:\s*(.*?)(?:\nDischarge Medications:|\n\Z)", re.DOTALL),
            "Discharge Medications": re.compile(r"Discharge Medications:\s*(.*?)(?:\nDischarge Disposition:|\n\Z)", re.DOTALL),
            "Discharge Diagnosis": re.compile(r"Discharge Diagnosis:\s*(.*?)(?:\nDischarge Condition:|\n\Z)", re.DOTALL),
        }

        # Model setup
        model_map = {
            'biobert': (biobert_model, biobert_tokenizer),
            'bluebert': (bluebert_model, bluebert_tokenizer),
            'pubmed': (pubmed_model, pubmed_tokenizer)
        }
        
        self.model, self.tokenizer = model_map.get(model_name, (None, None))
        if self.model:
            self.model = self.model.to(self.device)

    def extract_sections(self, text):
        """Extract clinical sections from discharge notes."""
        sections = {}
        for section, pattern in self.patterns.items():
            match = pattern.search(text)
            sections[section] = match.group(1).strip() if match else ""
        return sections

    def preprocess_text(self, text):
        """Clean text while preserving medical terms."""
        text = text.lower()
        text = re.sub(r"[^a-zA-Z0-9\s.,-]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def get_bert_embedding_batch(self, text_list, batch_size=64):
        """Generate embeddings efficiently in batches."""
        all_embeddings = []
        
        for i in tqdm(range(0, len(text_list), batch_size), desc="Generating Embeddings"):
            batch_texts = text_list[i:i + batch_size]
            inputs = self.tokenizer(batch_texts, padding=True, truncation=True, 
                                 return_tensors="pt", max_length=512)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
    
            with torch.no_grad():
                outputs = self.model(**inputs)
    
            batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(batch_embeddings)
    
        return np.vstack(all_embeddings)

    def store_faiss_index(self, embeddings, texts, metadatas, save_path):
        """Store precomputed embeddings in FAISS with metadata."""
        os.makedirs(save_path, exist_ok=True)
        
        def dummy_embedding_function(text: str):
            return [0.0] * embeddings.shape[1]
        
        text_embeddings = list(zip(texts, embeddings))
        
        db = FAISS.from_embeddings(
            text_embeddings=text_embeddings,
            embedding=dummy_embedding_function,
            metadatas=metadatas
        )
        db.save_local(save_path)

    def process_notes_parallel(self, save_faiss=False, faiss_path="discharge_embeddings"):
        """Process notes by sections, generate embeddings, and store in FAISS."""
        all_texts = []
        all_metadatas = []
        
        # Process each note and extract sections
        for _, row in tqdm(self.data.iterrows(), total=len(self.data), desc="Processing Notes"):
            subject_id = row['subject_id']
            note_id = row["note_id"]
            sections = self.extract_sections(row["text"])
            
            for section_name, section_text in sections.items():
                if section_text:  # Only process non-empty sections
                    processed_text = self.preprocess_text(section_text)
                    all_texts.append(processed_text)
                    all_metadatas.append({
                        "note_id": note_id,
                        "patient_id" : subject_id,
                        "section": section_name,
                        "original_text": section_text  # Store original unprocessed text
                    })
        
        # Generate embeddings for all sections
        embeddings = self.get_bert_embedding_batch(all_texts, batch_size=64)
        
        # Store in FAISS if requested
        if save_faiss:
            self.store_faiss_index(
                embeddings=embeddings,
                texts=all_texts,
                metadatas=all_metadatas,
                save_path=faiss_path
            )
        
        return embeddings, all_metadatas

In [12]:
embedding_generator = GenerateEmbedding(
    notes_sample, "biobert",
    biobert_model, biobert_tokenizer,
    # bluebert_model, bluebert_tokenizer,
    # pubmed_model, pubmed_tokenizer
)

# Generate embeddings and store in FAISS
embeddings = embedding_generator.process_notes_parallel(save_faiss=True)

Generating Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 107/107 [00:59<00:00,  1.81it/s]
`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [13]:
!nvidia-smi

Wed Apr 23 15:44:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           On  |   00000000:88:00.0 Off |                    0 |
| N/A   50C    P0             45W /  250W |    8728MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
class RetrieveAndGenerate:
    def __init__(self, faiss_path, llama_models, bert_model_name):
        self.llama_models = llama_models
        self.bert_model_name = bert_model_name
        bert_model_map = {
            "biobert": "dmis-lab/biobert-base-cased-v1.1",
            "bluebert": "bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16",
            "pubmed": "NeuML/pubmedbert-base-embeddings"
        }
 
        if bert_model_name not in bert_model_map:
            raise ValueError(f"Unsupported BERT model: {bert_model_name}. Choose from {list(bert_model_map.keys())}")
 
        self.embedding_function = HuggingFaceEmbeddings(model_name=bert_model_map[bert_model_name])
        self.faiss_index = FAISS.load_local(faiss_path,
                                            embeddings=self.embedding_function,
                                            allow_dangerous_deserialization=True)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    def retrieve_context(self, query, top_k, patient_id):
        """
        Retrieve top-k similar notes from FAISS index.
        If patient_id is provided, filter results using metadata.
        """
    
        # Get more than top_k so we can filter
        docs = self.faiss_index.similarity_search(query, k=top_k * 5)
    
        # Filter by patient_id metadata
        if patient_id is not None:
            docs = [doc for doc in docs if str(doc.metadata.get("patient_id")) == str(patient_id)]
            print(doc.metadata.get("patient_id"))
        
        # Trim to top_k
        docs = docs[:top_k]
    
        return [doc.page_content for doc in docs]
 
    def build_prompt(self, context, patient_id, mode="generate"):
        """Construct a structured and HIPAA-compliant prompt for clinical note generation."""
        base_instruction = (
            "You are a clinical documentation generator. Create a synthetic clinical note that follows HIPAA compliance. "
            "Do not include real names, exact dates, specific locations, or identifying numbers. "
            "Use placeholders like '[REDACTED]' **only** for personal identifiers. "
            "**Do not redact medical content** such as symptoms, conditions, exam findings, or treatments. "
            "Use general terms like 'the patient' instead of actual names.\n\n"
            "Structure your note as follows:\n"
 
            "**Patient's Name:** [REDACTED]\n"
            "**Age:** [REDACTED]\n"
            "**Sex:** <Male/Female>\n"
            "**Medical History:** <List chronic or relevant past conditions>\n\n"
            "**Chief Complaint:** <Summarize presenting problem>\n\n"
            "**History of Present Illness:** <Narrative of symptom progression>\n\n"
            "**Past Medical History:** <Summarize chronic illnesses, surgeries>\n\n"
            "**Physical Examination:** <Vital signs, physical findings>\n\n"
            "**Pertinent Results:** <Results of lab tests, imaging, etc.>\n\n"
            "**Discharge Diagnosis:** <Primary and secondary diagnoses>\n\n"
            "**Discharge Medications:** <List of prescribed medications on discharge>\n\n"
            "### Begin Note Below ###\n"
        )
        if mode == "validate" and patient_id is not None:
            return base_instruction + (context if context else "")
        else:
            return base_instruction + (context if context else "")
 
    def scrub_phi(self, text):
        # Redact patient name
        text = re.sub(r"(?i)(Patient's Name|Name):\s*.*\n", r"\1: [REDACTED]\n", text)
        # Redact specific ages (e.g., "Age: 32")
        text = re.sub(r"(?i)(Age):\s*\d+\b", r"\1: [REDACTED]", text)
        # Redact narrative references like "Ms. Rodriguez", "Mr. Smith", etc.
        text = re.sub(r"\b(Ms\.|Mr\.|Mrs\.|Dr\.)\s+\w+\b", "the patient", text)
        # Optionally remove height/weight if needed
        # text = re.sub(r"(Height|Weight|BMI):\s*.*", r"\1: [REDACTED]", text)
        return text
 
    def generate_synthetic_notes(self, query, llama_model_name, patient_id, mode):
        if llama_model_name not in self.llama_models:
            raise ValueError(f"LLaMA model '{llama_model_name}' not found in available models.")
        model, tokenizer = self.llama_models[llama_model_name]
        model = model.to(self.device)
        context = None
        if mode != "validate":
            if query is None:
                raise ValueError("Query must be provided in 'generate' mode.")
            retrieved_texts = self.retrieve_context(query=query, top_k=5, patient_id=patient_id)
            context = " ".join(retrieved_texts)
        else:
            # Optionally still pull context from patient-specific FAISS entries
            retrieved_texts = self.retrieve_context(query="Patient with Alzheimer's disease or dementia", top_k=5, patient_id=patient_id)
            context = " ".join(retrieved_texts) if retrieved_texts else None
        input_prompt = self.build_prompt(context=context, patient_id=patient_id, mode=mode)
        inputs = tokenizer(input_prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
                top_p=0.9,
                do_sample=False
            )
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        # note_only = decoded_output.split("### Begin Note Below ###")[-1].strip()
        match = re.search(r"### Begin Note Below ###(.*?)### End Note ###", decoded_output, re.DOTALL)
        note_only = match.group(1).strip() if match else ""
        cleaned_output = self.scrub_phi(note_only)
        return cleaned_output

In [19]:
# --- Initialize RetrieveAndGenerate ---
rag = RetrieveAndGenerate(
    faiss_path="biobert_embeddings",
    llama_models={"llama-3-3b": (llama3_model, llama3_tokenizer)},
    bert_model_name="biobert"  # choose "biobert", "bluebert", or "pubmed"
)

# --- Run in VALIDATION mode (note without including patient history in prompt) ---
validated_note = rag.generate_synthetic_notes(
    query="Patient with Alzheimer's disease or dementia",
    llama_model_name="llama-3-3b",
    patient_id=11968246,
    mode="validate"
)
print(validated_note)

No sentence-transformers model found with name dmis-lab/biobert-base-cased-v1.1. Creating a new one with mean pooling.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Patient's Name: [REDACTED]
**Age:** [REDATED]
**Sex:** [Male/Female]
**Medical History:** Hypertension, hyperlipidemia, type 2 diabetes mellitus

**Chief Complaint:** The patient presented with a complaint of worsening shortness of breath and fatigue.

**History of Present Illness:** The patient reported a 2-week history of progressive dyspnea and fatigue, which worsened over the past 3 days. The patient also reported a 10-pound weight loss in the past 2 weeks.

**Past Medical History:** The patient has a history of hypertension, hyperlipidemia, and type 2 diabetes mellitus.

**Physical Examination:** Vital signs: BP 140/90, HR 100, RR 18, O2 Sat 92% on room air. Physical examination revealed bilateral lung congestion and bilateral crepitus.

**Pertinent Results:** Chest X-ray showed bilateral infiltrates. ECG showed ST-segment depression.

**Discharge Diagnosis:** Acute exacerbation of chronic obstructive pulmonary disease (COPD) with pneumonia.

**Discharge Medications:** Albuterol

In [21]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Example data
generated_notes = validated_note

ground_truth_notes = notes_sample.loc[notes_sample['subject_id'] == 12784119, 'text'].values[0]

# ---------- ROUGE ----------
def compute_rouge(generated, reference):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = [scorer.score(r, g) for r, g in zip(reference, generated)]
    for i, s in enumerate(scores):
        print(f"\nNote {i + 1} ROUGE Scores:")
        for k, v in s.items():
            print(f"{k}: Precision={v.precision:.4f}, Recall={v.recall:.4f}, F1={v.fmeasure:.4f}")

compute_rouge(generated_notes, ground_truth_notes)


Note 1 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 2 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 3 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 4 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 5 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 6 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 7 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 8 ROUGE Scores:
rouge1: Precision=0.0000, Recall=0.0000, F1=0.0000
rougeL: Precision=0.0000, Recall=0.0000, F1=0.0000

Note 9 

In [ ]:
# ---------- BERTScore ----------
P, R, F1 = bert_score(generated_notes, ground_truth_notes, lang="en", rescale_with_baseline=True)

print("\nAverage BERTScore:")
print(f"Precision: {P.mean():.4f}")
print(f"Recall:    {R.mean():.4f}")
print(f"F1 Score:  {F1.mean():.4f}")